## Rigid body docking

In [ ]:
# Prepare coordinates files of monomers for docking

awk '$1=="ATOM" && substr($0,22,1)=="A"' 1A2P.pdb > 1a2pA.pdb

awk '$1=="ATOM" && substr($0,22,1)=="A"' 1A19.pdb > 1a19A.pdb

awk '$1=="ATOM" && substr($0,22,1)=="A"' 1BRS.pdb > 1brsA.pdb

awk '$1=="ATOM" && substr($0,22,1)=="D"' 1BRS.pdb > 1brsD.pdb


cat 1brsA.pdb > 1brsAD.pdb
cat 1brsD.pdb >> 1brsAD.pdb


# Running ZDOCK

cp /mnt/NFS_UPF/soft/zdock/IntelP3_Linux/uniCHARMM ./
cp /mnt/NFS_UPF/soft/zdock/IntelP3_Linux/create_lig ./


/mnt/NFS_UPF/soft/zdock/IntelP3_Linux/mark_sur 1A2P_A.pdb 1BRS_r.pdb
/mnt/NFS_UPF/soft/zdock/IntelP3_Linux/mark_sur 1A19_B.pdb 1BRS_l.pdb


/mnt/NFS_UPF/soft/zdock/IntelP3_Linux/zdock -R 1BRS_r.pdb -L 1BRS_l.pdb -o 1BRS_zdock.out


# Running FTDOCK

ftdock -static 1A2P_A.pdb -mobile 1A19_B.pdb -noelec -calculate_grid 1.2 -angle_step 12 -internal -15 -surface 1.3 -keep 3 -out 1BRS.ftdock



# Analysis of ZDOCK docking solutions

head -n 14 1BRS_zdock.out > 1BRS_zdock_10.out


/mnt/NFS_UPF/soft/zdock/IntelP3_Linux/create.pl 1BRS_zdock_10.out


arrange.pl complex.1 complex.1.pdb
arrange.pl complex.2 complex.2.pdb
arrange.pl complex.3 complex.3.pdb
arrange.pl complex.4 complex.4.pdb
arrange.pl complex.5 complex.5.pdb
arrange.pl complex.6 complex.6.pdb
arrange.pl complex.7 complex.7.pdb
arrange.pl complex.8 complex.8.pdb
arrange.pl complex.9 complex.9.pdb
arrange.pl complex.10 complex.10.pdb



# Analysis of FTDOCK docking solutions


build -in 1BRS.ftdock -b1 1 -b2 10


# Compare the top 10 dockings sotutions of both with references complex 

# Running PATCHDock

buildParams.pl antibody.pdb ovine.pdb 4.0 AA


patch_dock.Linux params.txt out_file1


#Generate top 10 docking solutions from PATCHDock

transOutput.pl out_file1 1 10


## Energy-based scoring and flexible refinement of docking runs


- pyDock (rigid-body)
    - FTDock
    - ZDock

- FireDock (allows certain flexibility)
    - PatchDock

In [ ]:
# Energy-based scoring with pyDock

## Prepare the input files

awk '$1=="ATOM" && substr($0,22,1)=="A"' 2HQS.pdb > 2hqs_A_H.pdb

awk '$1=="ATOM" && substr($0,22,1)=="H"' 2HQS.pdb >> 2hqs_A_H.pdb


## Creation of the `.ini` file

[receptor]
pdb = 1c5k.pdb
mol = A
newmol = A
[ligand]
pdb = 1oap.pdb
mol = A
newmol = B


## Run pyDock

pyDock3 T26 setup


## DOCKING USING ZDOCK (10-15min)

/mnt/NFS_UPF/soft/zdock/IntelP3_Linux/zdock -R T26_rec.pdb -L T26_lig.pdb -o T26.zdock


## Transform ZDOCK output to pyDock root file

pyDock3 T26 rotzdock

cp T26.rot T26.rot.backup

head -200 T26.rot.backup > T26.rot

pyDock3 T26 dockser


## Generate top10 docking poses based on pyDock rank

pyDock3 T26 makePDB 1 10


In [ ]:
# PatchDock - FireDock


## Rigid Docking using PatchDock (15min)

buildParams.pl savinase.pdb BASI.pdb 4.0 EI

patch_dock.Linux params.txt out_file2

transOutput.pl out_file2 1 10

## Refinement using FireDock

### FireDock uses protonates pdb files of the proteins!!!!!!!!!

/mnt/NFS_UPF/soft/FireDock/PatchDockOut2Trans.pl out_file2 > ex1.trans

head -200 ex1.trans > tmp
mv tmp ex1.trans

/mnt/NFS_UPF/soft/FireDock/buildFireDockParams.pl savinase.pdb.CHB.pdb BASI.pdb.CHB.pdb U U EI ex1.trans firedock.out 0 50 0.8 1 paramsREF.txt

## Execute Refinement (10-20min)

/mnt/NFS_UPF/soft/FireDock/runFireDock.pl paramsREF.txt

sort -n -k11 firedock.out.ref > firedock.out.ref_sorted

## Ab initio or unbiased docking

In [ ]:
## Add restrains that appear in the interacting region in the  `.ini` files

[receptor]
pdb = 1C5K.pdb
mol = A
newmol = A
restr = A.His.246,A.Ala.249,A.Thr.292

[ligand]
pdb = 1OAP.pdb
mol = A
newmol = B
restr = A.Ala.88,A.Phe.94,A.Ser.126.A.Gly.1288


pyDock3 T26 dockrst > dockrst.log &



